In [60]:
from langgraph.graph import StateGraph, END, START
from langchain_groq import ChatGroq 
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel,Field
import operator


In [61]:
load_dotenv()
model= ChatGroq(model='llama-3.3-70b-versatile')

In [62]:
class EvaluationSchema(BaseModel):
    feedback: str=Field(description='Detailed feedback for essay')
    score: int=Field(description='Scor out of 10',ge=0,le=10)
    


In [63]:
structured_output=model.with_structured_output(EvaluationSchema)


In [64]:
essay="""The Importance of Technology in Modern Life

Technology has become an integral part of modern society, influencing the way people live, work, and communicate. Over the past few decades, rapid advancements in digital tools and systems have transformed nearly every aspect of human life. From smartphones and the internet to artificial intelligence and cloud computing, technology continues to reshape the world in profound ways.

One of the most significant impacts of technology is in communication. People can now connect instantly across the globe through messaging apps, video calls, and social media platforms. This has not only improved personal relationships but also enabled businesses to operate more efficiently in a global market. Remote work, which was once uncommon, has now become a standard practice in many industries.

Education is another area where technology has brought remarkable changes. Online learning platforms, virtual classrooms, and digital resources have made education more accessible than ever before. Students can learn at their own pace, access a wide range of materials, and interact with instructors from different parts of the world. This flexibility has opened new opportunities for lifelong learning.

In addition, technology has greatly enhanced productivity and innovation. Automation tools and software systems help businesses streamline operations, reduce errors, and improve efficiency. Industries such as healthcare, finance, and transportation have benefited significantly from technological advancements, leading to better services and improved quality of life.

However, the growing reliance on technology also presents certain challenges. Issues such as data privacy, cybersecurity threats, and digital addiction have become increasingly important. It is essential for individuals and organizations to use technology responsibly and develop strategies to address these concerns.

In conclusion, technology plays a crucial role in shaping modern life. While it offers numerous benefits in terms of communication, education, and productivity, it also requires careful management to mitigate potential risks. As technology continues to evolve, its impact on society will only grow stronger, making it essential for people to adapt and use it wisely."""

In [65]:
prompt="""Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}"""
structured_output.invoke(prompt)

EvaluationSchema(feedback='The essay has a clear and concise structure, with well-organized paragraphs and proper use of transitional phrases. However, there are some minor errors in grammar and punctuation, and the vocabulary could be more diverse. Additionally, some sentences are a bit too long and convoluted, which can make them hard to follow. Overall, the language quality is good, but could be improved with some revisions.', score=7)

In [66]:
class UPSCState(TypedDict):
    essay: str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback: str
    individual_scores: Annotated[list[int],operator.add]
    avg_score: float

In [67]:
def evaluate_language(state:UPSCState):
    prompt=f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output=structured_output.invoke(prompt)
    return {'language_feedback':output.feedback, 'individual_scores': [output.score]}


In [68]:
def evaluate_analysis(state:UPSCState):
    prompt=f'Evaluate the depth of analyis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output=structured_output.invoke(prompt)
    return {'analysis_feedback':output.feedback, 'individual_scores': [output.score]}


In [69]:
def evaluate_thought(state:UPSCState):
    prompt=f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output=structured_output.invoke(prompt)
    return {'clarity_feedback':output.feedback, 'individual_scores': [output.score]}


In [71]:
def final_evaluation(state: UPSCState):
    prompt = f"Based on the following feedbacks create a summarized feedback\n language feedback - {state['language_feedback']}\n depth of analysis feedback - {state['analysis_feedback']} \n clarity of thoughts feedback - {state['clarity_feedback']}"
    overall_feedback = model.invoke(prompt).content
    
    # FIX THE PARENTHESES HERE
    avg = sum(state['individual_scores']) / len(state['individual_scores'])
    
    return {'overall_feedback': overall_feedback, 'avg_score': avg}


In [72]:
graph=StateGraph(UPSCState)
graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

graph.add_edge(START,'evaluate_language' )
graph.add_edge(START,'evaluate_analysis' )
graph.add_edge(START,'evaluate_thought' )

graph.add_edge('evaluate_language','final_evaluation' )
graph.add_edge('evaluate_analysis','final_evaluation' )
graph.add_edge('evaluate_thought','final_evaluation' )

graph.add_edge('final_evaluation',END)

workflow=graph.compile()


In [74]:
initial_state={
    'essay':essay
}
workflow.invoke(initial_state)

{'essay': 'The Importance of Technology in Modern Life\n\nTechnology has become an integral part of modern society, influencing the way people live, work, and communicate. Over the past few decades, rapid advancements in digital tools and systems have transformed nearly every aspect of human life. From smartphones and the internet to artificial intelligence and cloud computing, technology continues to reshape the world in profound ways.\n\nOne of the most significant impacts of technology is in communication. People can now connect instantly across the globe through messaging apps, video calls, and social media platforms. This has not only improved personal relationships but also enabled businesses to operate more efficiently in a global market. Remote work, which was once uncommon, has now become a standard practice in many industries.\n\nEducation is another area where technology has brought remarkable changes. Online learning platforms, virtual classrooms, and digital resources have